<a href="https://colab.research.google.com/github/shahdhesham/Thesis_Set2/blob/main/GPT_Set2_SelfRefine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai

In [ ]:

from openai import OpenAI
from google.colab import userdata

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)


In [ ]:
from google.colab import files
import zipfile
import torch

import os

In [ ]:
import shutil

# CLEANUP - Remove old folders before extraction
if os.path.exists('input_folder'):
    shutil.rmtree('input_folder')
if os.path.exists('output_folder'):
    shutil.rmtree('output_folder')


In [ ]:
import shutil
import os

# Force delete everything
for folder in ['input_folder', 'output_folder']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"✅ Deleted {folder}")

# Verify they're gone
print("\n📂 Current folders:")
!ls -la


📂 Current folders:
total 16
drwxr-xr-x 1 root root 4096 May 12 13:35 .
drwxr-xr-x 1 root root 4096 May 15 21:15 ..
drwxr-xr-x 4 root root 4096 May 12 13:35 .config
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data


In [ ]:
import shutil
import os

# Delete EVERYTHING (folders AND old zips)
!rm -rf input_folder output_folder *.zip

print("✅ All cleaned up!")
print("\n📂 Current directory:")
!ls -la

✅ All cleaned up!

📂 Current directory:
total 16
drwxr-xr-x 1 root root 4096 May 12 13:35 .
drwxr-xr-x 1 root root 4096 May 15 21:15 ..
drwxr-xr-x 4 root root 4096 May 12 13:35 .config
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data


In [ ]:
# 1. Upload ZIP file
print("Upload your ZIP file containing .c files:")
uploaded = files.upload()
zip_name = next(iter(uploaded))

# 2. Extract ZIP
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('input_folder')
print("Files extracted to 'input_folder/'")

Upload your ZIP file containing .c files:


Saving C.zip to C.zip
Files extracted to 'input_folder/'


In [ ]:
def translate_batch(c_code_list):
    results = []

    for i, c_code in enumerate(c_code_list):
        print(f"Processing {i+1}/{len(c_code_list)}...")

        system_prompt = """You are an expert code translator. Your ONLY task is to convert C code to C++ code.
Rules you MUST follow:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior."""

        user_prompt = f"""Translate this C code to C++ code:

C Code:
{c_code}

C++ Code:"""

        try:
            # ── STEP 1: Initial Translation ──────────────
            response1 = client.responses.create(
                model="gpt-5.1",
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt}
                ],
            )
            initial_translation = response1.output_text

            # ── STEP 2: Self-Refinement ───────────────────
            refinement_prompt = f"""Review your C++ translation against these rules:
1. Output ONLY executable C++ code
2. Never include markdown or explanations
3. Preserve all functionality exactly
4. Use standard C++ libraries
5. Match the original code's input/output behavior

Additionally verify:
- All logic and functionality from the original C code is preserved
- All variables, functions and structures are correctly translated
- The code would compile without errors in a standard C++ compiler

Your translation:
{initial_translation}

If your translation violates any rule, fix it and resubmit the corrected C++ code only.
If your translation is correct, resubmit it as is.

C++ Code:"""

            response2 = client.responses.create(
                model="gpt-5.1",
                input=[
                    {"role": "system",    "content": system_prompt},
                    {"role": "user",      "content": user_prompt},
                    {"role": "assistant", "content": initial_translation},
                    {"role": "user",      "content": refinement_prompt}
                ],
            )

            final_translation = response2.output_text
            final_translation = final_translation.replace("```cpp", "").replace("```c++", "").replace("```", "").strip()
            results.append(final_translation)

        except Exception as e:
            print(f"  ❌ Error on file {i+1}: {e}")
            results.append(f"// Translation failed: {e}")

    print(f"\n✅ Done. Total files processed: {len(results)}")
    return results

In [ ]:
#batching
batch_size = 4
batch_files = []
batch_codes = []
batch_paths = []

for root, _, files in os.walk('input_folder'):
    for file in files:
        if file.endswith('.c'):
            in_path = os.path.join(root, file)
            out_path = in_path.replace('input_folder', 'output_folder').replace('.c', '.cpp')
            os.makedirs(os.path.dirname(out_path), exist_ok=True)

            with open(in_path, 'r') as f:
                code = f.read()

            batch_files.append(file)
            batch_codes.append(code)
            batch_paths.append((in_path, out_path))

            # Once batch is full, translate all at once
            if len(batch_codes) == batch_size:
                translations = translate_batch(batch_codes)
                for (in_p, out_p), translation in zip(batch_paths, translations):
                    with open(out_p, 'w') as f_out:
                        f_out.write(translation)
                    print(f"Translated: {in_p} → {out_p}")


                batch_files = []
                batch_codes = []
                batch_paths = []




# Translate any remaining files smaller than batch size
if batch_codes:
    translations = translate_batch(batch_codes)
    for (in_p, out_p), translation in zip(batch_paths, translations):
        with open(out_p, 'w') as f_out:
            f_out.write(translation)
        print(f"Translated: {in_p} → {out_p}")

Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/2048.c → output_folder/C/2048.cpp
Translated: input_folder/C/13545.c → output_folder/C/13545.cpp
Translated: input_folder/C/723.c → output_folder/C/723.cpp
Translated: input_folder/C/7320.c → output_folder/C/7320.cpp
Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/13653.c → output_folder/C/13653.cpp
Translated: input_folder/C/8947.c → output_folder/C/8947.cpp
Translated: input_folder/C/12814.c → output_folder/C/12814.cpp
Translated: input_folder/C/2074.c → output_folder/C/2074.cpp
Processing 1/4...
Processing 2/4...
Processing 3/4...
Processing 4/4...

✅ Done. Total files processed: 4
Translated: input_folder/C/13513.c → output_folder/C/13513.cpp
Translated: input_folder/C/9367.c → output_folder/C/9367.cpp
Translated: input_folder/C/139.c → output_folder/C/139.cpp
Transla

In [ ]:
from google.colab import files as colab_files  # CHANGED: Added alias

In [ ]:
# 6. Compress and download
print("\nCreating output ZIP...")
!zip -r output.zip output_folder
colab_files.download('output.zip')  # CHANGED: Uses alias
print("Done! Download should start automatically.")


Creating output ZIP...
  adding: output_folder/ (stored 0%)
  adding: output_folder/C/ (stored 0%)
  adding: output_folder/C/13414.cpp (deflated 73%)
  adding: output_folder/C/1710.cpp (deflated 53%)
  adding: output_folder/C/13537.cpp (deflated 32%)
  adding: output_folder/C/264.cpp (deflated 76%)
  adding: output_folder/C/2142.cpp (deflated 50%)
  adding: output_folder/C/9094.cpp (deflated 35%)
  adding: output_folder/C/1973.cpp (deflated 67%)
  adding: output_folder/C/8760.cpp (deflated 51%)
  adding: output_folder/C/1623.cpp (deflated 61%)
  adding: output_folder/C/1687.cpp (deflated 51%)
  adding: output_folder/C/10557.cpp (deflated 32%)
  adding: output_folder/C/2538.cpp (deflated 62%)
  adding: output_folder/C/1705.cpp (deflated 60%)
  adding: output_folder/C/9703.cpp (deflated 52%)
  adding: output_folder/C/2.cpp (deflated 55%)
  adding: output_folder/C/2001.cpp (deflated 50%)
  adding: output_folder/C/9497.cpp (deflated 52%)
  adding: output_folder/C/7321.cpp (deflated 57%)
 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Download should start automatically.
